In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class FashionDataLoader:
    def __init__(self, file_path):
        self.data = pd.read_csv(file_path)
        # Fashion-MNIST Label mapping
        self.label_names = {
            0: "T-shirt/top", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
            5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle boot"
        }

    def prepare_data(self):
        data = np.array(self.data)
        np.random.shuffle(data)
        
        # Validation set (1000 samples)
        data_dev = data[0:1000].T
        Y_dev = data_dev[0]
        X_dev = data_dev[1:785] / 255.0

        # Training set
        data_train = data[1000:].T
        Y_train = data_train[0]
        X_train = data_train[1:785] / 255.0

        return X_train, Y_train, X_dev, Y_dev

In [11]:
class FashionNet:
    def __init__(self, input_size=784, hidden_size=10, output_size=10):
        # Professional weight initialization
        self.params = {
            'W1': np.random.randn(hidden_size, input_size) * np.sqrt(2./input_size),
            'b1': np.zeros((hidden_size, 1)),
            'W2': np.random.randn(output_size, hidden_size) * np.sqrt(2./hidden_size),
            'b2': np.zeros((output_size, 1))
        }

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return Z > 0

    def softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

    def one_hot(self, Y):
        one_hot_Y = np.zeros((Y.size, Y.max() + 1))
        one_hot_Y[np.arange(Y.size), Y] = 1
        return one_hot_Y.T

    def forward_propagation(self, X):
        cache = {}
        cache['Z1'] = self.params['W1'].dot(X) + self.params['b1']
        cache['A1'] = self.relu(cache['Z1'])
        cache['Z2'] = self.params['W2'].dot(cache['A1']) + self.params['b2']
        cache['A2'] = self.softmax(cache['Z2'])
        return cache

    def backward_propagation(self, cache, X, Y):
        m = X.shape[1]
        Y_encoded = self.one_hot(Y)

        # Gradients for Output Layer
        dZ2 = cache['A2'] - Y_encoded
        dW2 = 1/m * dZ2.dot(cache['A1'].T)
        db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)

        # Gradients for Hidden Layer
        dZ1 = self.params['W2'].T.dot(dZ2) * self.relu_derivative(cache['Z1'])
        dW1 = 1/m * dZ1.dot(X.T)
        db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)

        return dW1, db1, dW2, db2

    def train(self, X, Y, iterations, alpha):
        for i in range(iterations):
            cache = self.forward_propagation(X)
            dW1, db1, dW2, db2 = self.backward_propagation(cache, X, Y)
            
            # Updating parameters using Gradient Descent
            self.params['W1'] -= alpha * dW1
            self.params['b1'] -= alpha * db1
            self.params['W2'] -= alpha * dW2
            self.params['b2'] -= alpha * db2
            
            if i % 100 == 0:
                predictions = np.argmax(cache['A2'], axis=0)
                accuracy = np.sum(predictions == Y) / Y.size
                print(f"Iteration {i}: Training Accuracy {accuracy:.4f}")

    def predict(self, X):
        cache = self.forward_propagation(X)
        return np.argmax(cache['A2'], axis=0)

In [ ]:
# 1. Load and Prepare Data
loader = FashionDataLoader('fashion-mnist_train.csv')
X_train, Y_train, X_dev, Y_dev = loader.prepare_data()

# 2. Initialize and Train (Using 20 neurons in the hidden layer for better results)
model = FashionNet(hidden_size=20)
model.train(X_train, Y_train, iterations=500, alpha=0.15)

# 3. Test on a random image from the validation set
def visualize_prediction(index):
    img = X_dev[:, index, None]
    prediction = model.predict(img)
    true_label = Y_dev[index]
    
    print(f"Model Prediction: {loader.label_names[prediction[0]]}")
    print(f"True Category: {loader.label_names[true_label]}")
    
    plt.imshow(img.reshape(28, 28), cmap='gray')
    plt.title(f"Predicted: {loader.label_names[prediction[0]]}")
    plt.show()

visualize_prediction(np.random.randint(0, 1000))